In [1]:
"""
Stage 4 Assembly -- 08: Preflight
===================================
Final reconciliation before any model touches this data. Runs against the
splits produced by 05_splits.ipynb, AFTER 07_drop_binaries.ipynb has removed
the five binary regime indicators from every assembled table and split.

This notebook is complementary to, not a duplicate of, 06_floor_diagnostic.
06_floor_diagnostic checks one internal mechanism of robust_expanding_zscore
(the denominator floor) against PRE-normalisation Stage 3/1.5 inputs, before
union, assembly, targets, or splitting exist. This notebook checks the
FINISHED product -- the split parquets a model actually loads -- and has no
overlap with what 06 covers: it never touches the floor/sigma_f/Welford
internals, and 06 never touches the taxonomy, split files, or cross-dataset
consistency.

FOUR CHECKS
-----------
1. Taxonomy <-> parquet reconciliation, both directions, every split, every
   dataset. Every surviving model feature has exactly one taxonomy row;
   every taxonomy row corresponds to a column that actually exists. Since
   this runs after 07_drop_binaries, the five binaries should be entirely
   absent from both sides -- if any turn up, 07 either wasn't run or was
   run against a different copy of the data than 05_splits produced.

2. agg_means feature columns == panel feature columns. Validates the
   "one taxonomy file serves both agg_means and panel" assumption that
   03_assemble_panel.ipynb and the sparse KAN/MLP loaders both rely on,
   rather than assuming it holds.

3. Zero-fill mass per feature (post-clip, post-fillna(0.0)). A feature
   heavily imputed to exactly 0.0 puts a spike of fabricated mass at the
   spline grid's centre bin, which no diagnostic upstream of this point
   (including 06_floor_diagnostic, which runs on pre-fill data) would catch.

4. Per-feature std of final clipped values. Near-constant features consume
   a spline (or an edge in the sparse model) and contribute nothing --
   cheap to catch here, expensive to notice later as "why is this edge
   dead."

Also records final n_features / n_subthemes / n_themes per dataset, needed
to size the dense KAN/MLP hidden layers to match the sparse model's
subtheme/theme counts.

Nothing is modified. This notebook only reads and reports.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.config import OUT

ASM_DIR = OUT / '02_assembled'
SPLIT_DIR = OUT / '04_splits'
THEMES_DIR = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes')

SPLITS = ['Split_A', 'Split_B', 'Split_C', 'Split_D']
PARTS = ['train', 'val', 'test']

# Meta columns per dataset -- everything else is a feature.
META = {
    'agg_means': {'date', 'target_daily_return', 'minret_5d_pct', 'y_binary'},
    'agg_full_moments': {'date', 'target_daily_return', 'minret_5d_pct', 'y_binary'},
    'panel': {'permno', 'date', 'dlyret', 'dlycap', 'minret_5d_pct', 'y_binary'},
}

TAXONOMY_FILE = {
    'agg_means': 'numbered_classified_moment_inventory_means_only.csv',
    'agg_full_moments': 'numbered_classified_moment_inventory_long.csv',
    'panel': 'numbered_classified_moment_inventory_means_only.csv',
}

ZERO_FILL_WARN_PCT = 5.0   # flag any feature above this % exactly-zero
LOW_STD_WARN = 0.10        # flag any feature at or below this final std

# The five binaries were removed in 07_drop_binaries.ipynb. They are checked
# for by name here as a POSITIVE assertion that they are gone -- not treated
# as an expected/tolerated exception the way an earlier draft of this
# notebook did (when it ran before 07 existed). If any of these turn up in
# Check 1, that is a genuine failure: 07 either was not run, or was run
# against a different copy of the data than what 05_splits produced here.
DROPPED_BINARIES = {'vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
                    'curve_inverted_3m10y', 'credit_stress'}


def load_taxonomy(dataset: str) -> set:
    path = THEMES_DIR / TAXONOMY_FILE[dataset]
    df = pd.read_csv(path, dtype={'theme_id': str, 'subtheme_id': str})
    return set(df['column'])


# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 1: TAXONOMY <-> PARQUET RECONCILIATION
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 100)
print("CHECK 1: TAXONOMY <-> PARQUET RECONCILIATION")
print("=" * 100)

check1_clean = True
binaries_found = []
final_counts = {}   # dataset -> {'n_features', 'n_subthemes', 'n_themes'}

for dataset in META:
    taxonomy_cols = load_taxonomy(dataset)

    for split in SPLITS:
        for part in PARTS:
            path = SPLIT_DIR / split / f'{dataset}_{part}.parquet'
            cols = set(pd.read_parquet(path).columns)
            features = cols - META[dataset]

            in_data_not_taxonomy = features - taxonomy_cols
            in_taxonomy_not_data = taxonomy_cols - features

            # Binaries should be entirely absent post-07. Report their
            # presence as a genuine failure, not an expected/tolerated
            # exception -- that tolerance belonged to a pre-07 version of
            # this notebook and no longer applies.
            leaked_binaries = in_data_not_taxonomy & DROPPED_BINARIES
            unexplained_extra = in_data_not_taxonomy - DROPPED_BINARIES

            if leaked_binaries:
                check1_clean = False
                binaries_found.append((split, dataset, part, sorted(leaked_binaries)))
                print(f"  \u2717 {split}/{dataset}_{part}: dropped binaries "
                      f"still present: {sorted(leaked_binaries)}")

            if unexplained_extra:
                check1_clean = False
                print(f"  \u2717 {split}/{dataset}_{part}: "
                      f"{len(unexplained_extra)} feature(s) in data with no "
                      f"taxonomy row: {sorted(unexplained_extra)[:10]}")

            if in_taxonomy_not_data:
                check1_clean = False
                print(f"  \u2717 {split}/{dataset}_{part}: "
                      f"{len(in_taxonomy_not_data)} taxonomy row(s) with no "
                      f"matching column: {sorted(in_taxonomy_not_data)[:10]}")

    # Final counts taken from Split_D/train, since it has the fullest
    # coverage window and every dataset is present there.
    ref_path = SPLIT_DIR / 'Split_D' / f'{dataset}_train.parquet'
    ref_cols = set(pd.read_parquet(ref_path).columns) - META[dataset]
    tax_df = pd.read_csv(THEMES_DIR / TAXONOMY_FILE[dataset],
                        dtype={'theme_id': str, 'subtheme_id': str})
    tax_df = tax_df[tax_df['column'].isin(ref_cols)]
    final_counts[dataset] = {
        'n_features': len(ref_cols),
        'n_subthemes': tax_df['subtheme_id'].nunique(),
        'n_themes': tax_df['theme_id'].nunique(),
    }

if check1_clean:
    print(f"\n  \u2713 No taxonomy/data mismatches, and no dropped binaries "
          f"remain, across {len(META)} datasets x {len(SPLITS)} splits x "
          f"{len(PARTS)} parts")
else:
    if binaries_found:
        print(f"\n  \u2717 {len(binaries_found)} split/dataset/part "
              f"combination(s) still contain a supposedly-dropped binary. "
              f"Re-check that 07_drop_binaries.ipynb was run against THIS "
              f"copy of {SPLIT_DIR}.")
    print(f"\n  \u2717 Unexplained mismatches found -- see above. Investigate "
          f"before proceeding.")

print(f"\n  Final feature/subtheme/theme counts (from Split_D/train):")
for dataset, c in final_counts.items():
    print(f"    {dataset:<20} {c['n_features']:>5} features   "
          f"{c['n_subthemes']:>4} subthemes   {c['n_themes']:>3} themes")


# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 2: agg_means AND panel SHARE THE SAME FEATURE SET
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 100)
print("CHECK 2: agg_means AND panel FEATURE-SET IDENTITY")
print("=" * 100)

check2_clean = True

for split in SPLITS:
    for part in PARTS:
        means_cols = set(pd.read_parquet(
            SPLIT_DIR / split / f'agg_means_{part}.parquet').columns
        ) - META['agg_means']
        panel_cols = set(pd.read_parquet(
            SPLIT_DIR / split / f'panel_{part}.parquet').columns
        ) - META['panel']

        only_means = means_cols - panel_cols
        only_panel = panel_cols - means_cols

        if only_means or only_panel:
            check2_clean = False
            print(f"  \u2717 {split}/{part}: agg_means-only "
                  f"{sorted(only_means)[:5]}   panel-only "
                  f"{sorted(only_panel)[:5]}")

if check2_clean:
    print(f"\n  \u2713 agg_means and panel carry identical feature sets in "
          f"every split/part")
else:
    print(f"\n  \u2717 Feature sets diverge -- see above. This breaks the "
          f"assumption that one taxonomy file serves both datasets.")


# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 3: ZERO-FILL MASS PER FEATURE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 100)
print(f"CHECK 3: ZERO-FILL MASS (post-clip, post-fillna) -- flag > "
      f"{ZERO_FILL_WARN_PCT}%")
print("=" * 100)

# Checked on the training split of each dataset in Split_D, which has the
# largest and most complete training window.
zero_fill_flags = []

for dataset in META:
    path = SPLIT_DIR / 'Split_D' / f'{dataset}_train.parquet'
    df = pd.read_parquet(path)
    features = [c for c in df.columns if c not in META[dataset]]

    pct_zero = (df[features] == 0.0).mean() * 100
    flagged = pct_zero[pct_zero > ZERO_FILL_WARN_PCT].sort_values(ascending=False)

    for feat, pct in flagged.items():
        zero_fill_flags.append({'dataset': dataset, 'feature': feat, 'pct_zero': pct})

if zero_fill_flags:
    flags_df = pd.DataFrame(zero_fill_flags).sort_values('pct_zero', ascending=False)
    print(f"\n  {len(flags_df)} feature(s) above {ZERO_FILL_WARN_PCT}% exactly "
          f"zero (Split_D/train):")
    print(flags_df.head(30).to_string(index=False))
    if len(flags_df) > 30:
        print(f"  ... and {len(flags_df) - 30} more")
else:
    print(f"\n  \u2713 No feature exceeds {ZERO_FILL_WARN_PCT}% zero-fill mass "
          f"in Split_D/train")


# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 4: PER-FEATURE STD OF FINAL CLIPPED VALUES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 100)
print(f"CHECK 4: NEAR-CONSTANT FEATURES -- flag std <= {LOW_STD_WARN}")
print("=" * 100)

low_std_flags = []

for dataset in META:
    path = SPLIT_DIR / 'Split_D' / f'{dataset}_train.parquet'
    df = pd.read_parquet(path)
    features = [c for c in df.columns if c not in META[dataset]]

    stds = df[features].std()
    flagged = stds[stds <= LOW_STD_WARN].sort_values()

    for feat, s in flagged.items():
        low_std_flags.append({'dataset': dataset, 'feature': feat, 'std': s})

if low_std_flags:
    flags_df = pd.DataFrame(low_std_flags).sort_values('std')
    print(f"\n  {len(flags_df)} feature(s) at or below std={LOW_STD_WARN} "
          f"(Split_D/train):")
    print(flags_df.head(30).to_string(index=False))
    if len(flags_df) > 30:
        print(f"  ... and {len(flags_df) - 30} more")
else:
    print(f"\n  \u2713 No feature at or below std={LOW_STD_WARN} in "
          f"Split_D/train")


# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

print(f"""
  Check 1 (taxonomy/parquet reconciliation, incl. binaries gone): {'PASS' if check1_clean else 'FAIL'}
  Check 2 (agg_means == panel feature set):                        {'PASS' if check2_clean else 'FAIL'}
  Check 3 (zero-fill mass):                                        {len(zero_fill_flags)} feature(s) flagged
  Check 4 (near-constant features):                                {len(low_std_flags)} feature(s) flagged

  Final counts (Split_D/train, taxonomy-matched columns only):
""")
for dataset, c in final_counts.items():
    print(f"    {dataset:<20} {c['n_features']:>5} features   "
          f"{c['n_subthemes']:>4} subthemes   {c['n_themes']:>3} themes")

all_pass = check1_clean and check2_clean
print(f"""
  {'\u2713 ALL STRUCTURAL CHECKS PASSED -- data is ready for model code.' if all_pass
    else '\u2717 STRUCTURAL CHECKS FAILED -- see Check 1 / Check 2 above before proceeding.'}

  Checks 3 and 4 are advisory (flag candidates for review, not hard
  failures). Review any flagged features before finalising a large training
  run, but they do not block moving forward.
""")

CHECK 1: TAXONOMY <-> PARQUET RECONCILIATION

  ✓ No taxonomy/data mismatches, and no dropped binaries remain, across 3 datasets x 4 splits x 3 parts

  Final feature/subtheme/theme counts (from Split_D/train):
    agg_means              574 features    128 subthemes    13 themes
    agg_full_moments      1699 features    331 subthemes    13 themes
    panel                  574 features    128 subthemes    13 themes

CHECK 2: agg_means AND panel FEATURE-SET IDENTITY

  ✓ agg_means and panel carry identical feature sets in every split/part

CHECK 3: ZERO-FILL MASS (post-clip, post-fillna) -- flag > 5.0%

  21 feature(s) above 5.0% exactly zero (Split_D/train):
dataset               feature  pct_zero
  panel  MomOffSeason11YrPlus 10.610315
  panel CompositeDebtIssuance  9.790832
  panel       RevenueSurprise  9.071321
  panel                grcapx  8.994028
  panel               IntanBM  8.822731
  panel            Investment  8.732904
  panel      AbnormalAccruals  8.643972
  panel    